SELECT–FROM–GROUP BY

In [1]:
from sqlalchemy import create_engine, text
import pandas as pd
from pathlib import Path
import re
from typing import List, Set, Tuple, Dict, Optional, Any
import os, uuid
import numpy as np

from sqlalchemy.orm.base import PASSIVE_OFF


In [2]:
USER = "postgres"
HOST = "localhost"
PORT = "5432"
PASSWORD = "user"

DB_ADMIN_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/postgres"
engine_admin = create_engine(DB_ADMIN_URL, isolation_level="AUTOCOMMIT")

DB_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/synthea"
engine = create_engine(DB_URL)
print("Connesso al database synthea")

Connesso al database synthea


In [3]:
def run(sql_or_text, show=False):
    with engine.begin() as conn:
        stmt = text(sql_or_text) if isinstance(sql_or_text, str) else sql_or_text
        result = conn.execute(stmt)
        if result.returns_rows:
            df = pd.DataFrame(result.fetchall(), columns=result.keys())
            if show:
                display(df)
            return df
        return None


def _strip_semicolon(sql: str) -> str:
    return re.sub(r';\s*$', '', sql.strip())

def _unqualify(tok: str) -> str:
    tok = tok.strip().strip('"')
    return tok.split('.')[-1].lower() if '.' in tok else tok.lower()


def _split_outside_parents(s: str) -> List[str]:
    items, buf, d = [], [], 0
    for ch in s:
        if ch == '(':
            d += 1
        elif ch == ')':
            d = max(0, d - 1)
        if ch == ',' and d == 0:
            items.append(''.join(buf).strip())
            buf = []
        else:
            buf.append(ch)
    if buf:
        items.append(''.join(buf).strip())
    return items

In [4]:
def _count_table(tname: str) -> int:
    return int(run(f"SELECT COUNT(*) AS n FROM {tname};").iloc[0]["n"])


def _size_table(tname: str) -> int:
    return int(run(f"SELECT pg_total_relation_size('{tname}') AS bytes;").iloc[0]["bytes"])


def _table_cols(tname: str) -> list[str]:

    schema, table = tname.split(".", 1)
    df = run(f"""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = '{schema}' AND table_name = '{table}'
        ORDER BY ordinal_position;
    """)
    return [] if df.empty else df["column_name"].tolist()

def _avg_row_bytes(tname: str) -> int:
    cols = _table_cols(tname)
    if not cols:
        return 0
    expr = " + ".join([f"COALESCE(pg_column_size({c}),0)" for c in cols])
    df = run(f"SELECT COALESCE(AVG({expr})::bigint, 0) AS avg_bytes FROM {tname};")
    return int(df.iloc[0]["avg_bytes"])

def _payload_bytes(tname: str) -> int:

    rows = _count_table(tname)
    return rows * _avg_row_bytes(tname)

def _network_bytes_payload(ro_name: str | None, rs_name: str | None) -> int:
    total = 0
    if ro_name:
        total += _payload_bytes(ro_name)
    if rs_name:
        total += _payload_bytes(rs_name)
    return total

def _token_used(sql: str | None, token: str) -> bool:
    if not sql: return False
    # match "Ro" / "Rs" come identificatore o alias, non come sottostringa
    return re.search(rf'(?<!\w){token}(?!\w)', sql) is not None

def _ships(plan: dict) -> tuple[bool, bool]:
    # Ro viaggia se qs usa Ro; Rs viaggia se qso usa Rs
    return _token_used(plan.get("qs"), "Ro"), _token_used(plan.get("qso"), "Rs")

def _avg_payload_on(table_qual: str, cols: list[str]) -> int:
    if not cols: return 0
    expr = " + ".join([f"COALESCE(pg_column_size({c}),0)" for c in cols])
    df = run(f"SELECT COALESCE(AVG({expr})::bigint,0) AS b FROM {table_qual};")
    return int(df.iloc[0]["b"])

def _row_count_base(table_qual: str) -> int:
    df = run(f"SELECT COUNT(*) AS n FROM {table_qual};")
    return int(df.iloc[0]["n"])

def _num_groups(table_qual: str, keys: list[str]) -> int:
    if not keys: return 1
    cols = ", ".join(keys)
    df = run(f"SELECT COUNT(*) AS g FROM (SELECT DISTINCT {cols} FROM {table_qual}) t;")
    return int(df.iloc[0]["g"])




FRAMMENTAZIONE VERTICALE

# PATIENTS
Owner(PATIENTS) = { id, birthdate,ssn, drivers, passport, first, middle, last, maiden, address, city, fips, zip, lat, lon, income }


Server(PATIENTS) = { id, deathdate, gender, race, ethnicity, marital, prefix, suffix, birthplace, state, county, healthcare_expanses, healthcare_coverage}

In [5]:
# se servisse ricaricare i dati

sql = open("sql/fragmentPatients.sql").read()

with engine.begin() as conn:
    if sql.strip():
        conn.execute(text(sql))
        print("Frammentazione creata")
    else:
        print("Errore")

# O semplicemente uso la funzione run(sql)

Frammentazione creata


In [6]:
run(''' ANALYZE owner.patients_owner; ANALYZE server.patients_server;''')

In [7]:
Fo = {
    "id", "birthdate", "ssn", "drivers", "passport",
    "first", "middle", "last", "maiden",
    "address", "city", "fips", "zip", "lat", "lon",
    "income"
}

Fs = {
    "id", "deathdate", "gender", "race", "ethnicity", "marital",
    "prefix", "suffix", "birthplace", "state", "county",
    "healthcare_expenses", "healthcare_coverage"
}


In [8]:

def parse_query_groupby(query: str) -> Tuple[Set[str], Set[str], List[Dict[str, Any]]]:

    q = query.strip()
    m_sel = re.search(r"\bselect\s+(.*?)\s+from\b", q, re.I | re.S)
    if not m_sel:
        raise ValueError("SELECT ... FROM mancante.")
    sel_txt = m_sel.group(1)
    rest = q[m_sel.end():]


    m_gb = re.search(r"\bgroup\s+by\b", rest, re.I)


    group_by_txt = None
    if m_gb:
        group_by_txt = rest[m_gb.end():].strip()
        group_by_txt = re.sub(r';\s*$', '', group_by_txt, flags=re.S)

    # SELECT: separo plain vs aggregazioni
    select_items = _split_outside_parents(sel_txt)
    select_plain: Set[str] = set()
    aggs: List[Dict[str, Any]] = []

    agg_re = re.compile(
        r"^(count|sum|avg|min|max)\s*\(\s*(distinct\s+)?(\*|[a-zA-Z_][\w\.]*)\s*\)\s*(?:as\s+([a-zA-Z_]\w*))?$",
        re.I
    )
    for it in select_items:
        it_norm = it.strip()
        m = agg_re.match(it_norm)
        if m:
            func = m.group(1).lower()
            distinct = bool(m.group(2))
            arg_raw = m.group(3)
            alias = m.group(4).lower() if m.group(4) else None
            arg = None if arg_raw == '*' else _unqualify(arg_raw)
            aggs.append({"func": func, "arg": arg, "distinct": distinct, "alias": alias})
        else:

            select_plain.add(_unqualify(it_norm))

    group_by: Set[str] = set()
    if group_by_txt:
        cols = [tok for tok in _split_outside_parents(group_by_txt) if tok.strip()]
        group_by = {_unqualify(c) for c in cols}


    if select_plain - group_by:
        missing = ", ".join(sorted(select_plain - group_by))
        raise ValueError(f"Le colonne non aggregate in SELECT devono apparire nel GROUP BY (manca: {missing}).")

    return select_plain, group_by, aggs





def classify_groupby_agg(group_by: Set[str], aggs: List[Dict[str, Any]],
                         Fo: Set[str], Fs: Set[str]) -> Dict[str, Set[str]]:

    G_owner = {g for g in group_by if g in Fo}
    G_server = {g for g in group_by if g in Fs}
    Agg_owner = {a["arg"] for a in aggs if a.get("arg") and a["arg"] in Fo}
    Agg_server = {a["arg"] for a in aggs if a.get("arg") and a["arg"] in Fs}
    return {
        "G_owner": G_owner, "G_server": G_server,
        "Agg_owner": Agg_owner, "Agg_server": Agg_server,
    }





In [9]:
def choose_strategy_groupby(classified: dict,
                                   aggs: list[dict],
                                   fo_table: str = "owner.patients_owner",
                                   fs_table: str = "server.patients_server",
                                   reduction_threshold: float = 0.2) -> str:


    G_o, G_s = classified["G_owner"], classified["G_server"]
    A_o, A_s = classified["Agg_owner"], classified["Agg_server"]


    if not G_s and not A_s:
        return "owner-only"
    if not G_o and not A_o:
        return "server-only"

    if not G_o and G_s and A_o and not A_s:
        N = _row_count_base(fs_table)
        Gcount = _num_groups(fs_table, sorted(G_s)) if G_s else N
        return "server-owner" if (Gcount / max(N, 1)) <= reduction_threshold else "owner-server"


    if not G_s and G_o and A_s and not A_o:
        N = _row_count_base(fo_table)
        Gcount = _num_groups(fo_table, sorted(G_o)) if G_o else N
        return "owner-server" if (Gcount / max(N, 1)) <= reduction_threshold else "server-owner"

    score = []
    if G_o:
        N_o = _row_count_base(fo_table)
        Go = _num_groups(fo_table, sorted(G_o))
        score.append(("owner-first", Go / max(N_o, 1)))
    if G_s:
        N_s = _row_count_base(fs_table)
        Gs = _num_groups(fs_table, sorted(G_s))
        score.append(("server-first", Gs / max(N_s, 1)))

    if score:
     best = min(score, key=lambda x: x[1])
     if best[1] <= reduction_threshold:
        return "owner-server" if best[0] == "owner-first" else "server-owner"

    owner_weight = len(G_o) + len(A_o)
    server_weight = len(G_s) + len(A_s)
    if owner_weight > server_weight:
      return "owner-server"
    if server_weight > owner_weight:
        return "server-owner"

    return "parallel"


In [10]:

def render_aggs_sql(aggs: List[Dict[str, Any]], Fo: Set[str]) -> str:
    exprs = []
    for a in aggs:
        func = a["func"].upper()
        distinct = "DISTINCT " if a.get("distinct") else ""
        arg = a.get("arg")
        if arg is None:
            expr = f"{func}(*)"
            alias = a.get("alias") or f"{func.lower()}_all"
        else:
            qual = "o" if arg in Fo else "s"
            expr = f"{func}({distinct}{qual}.{arg})"
            alias = a.get("alias") or f"{func.lower()}_{arg}"
        exprs.append(f"{expr} AS {alias}")
    return ", ".join(exprs)



In [66]:
def generate_subqueries_gb(
        select_plain: Set[str], group_by: Set[str], aggs: List[Dict[str, Any]],
        Fo: Set[str], Fs: Set[str], strategy: str
) -> Tuple[str | None, str | None, str | None]:


    sel_plain = {c.lower() for c in select_plain}
    gb = {c.lower() for c in group_by}
    agg_args = {a["arg"] for a in aggs if a.get("arg")}

    need_fs = ((sel_plain | gb | agg_args) & Fs) - {'id'}
    need_fo = ((sel_plain | gb | agg_args) & Fo) - {'id'}

    Aqs = sorted(need_fs)
    Aqo = sorted(need_fo)

    gb_owner = [f"o.{c}" for c in sorted(gb & Fo)]
    gb_server = [f"s.{c}" for c in sorted(gb & Fs)]
    gb_all = gb_owner + gb_server
    gb_sql = ", ".join(gb_all)

    aggs_sql = render_aggs_sql(aggs, Fo)
    select_parts = []
    if gb_sql:
        select_parts.append(gb_sql)
    if aggs_sql:
        select_parts.append(aggs_sql)
    final_select = ", ".join(select_parts) if select_parts else aggs_sql

    qs = qo = qso = None

    if strategy == "server-owner":

        proj_qs = ", ".join(["s.id"] + [f"s.{c}" for c in Aqs])
        qs = f"SELECT {proj_qs} FROM server.patients_server s"


        qso = f"SELECT {final_select} FROM owner.patients_owner o JOIN Rs s USING (id)"
        if gb_sql:
            qso += f" GROUP BY {gb_sql}"

    elif strategy == "owner-server":

        qo = "SELECT o.id FROM owner.patients_owner o"


        proj_qs = ", ".join(["s.id"] + [f"s.{c}" for c in Aqs])
        qs = f"SELECT {proj_qs} FROM server.patients_server s JOIN Ro r USING (id)"

        qso = f"SELECT {final_select} FROM owner.patients_owner o JOIN Rs s USING (id)"
        if gb_sql:
            qso += f" GROUP BY {gb_sql}"

    elif strategy == "owner-only":

        needs_server = bool(((sel_plain | gb | agg_args) & Fs))
        if needs_server:
            qo = "SELECT o.id FROM owner.patients_owner o"
            proj_qs = ", ".join(["s.id"] + [f"s.{c}" for c in Aqs])
            qs = f"SELECT {proj_qs} FROM server.patients_server s JOIN Ro r USING (id)"
            qso = f"SELECT {final_select} FROM owner.patients_owner o JOIN Rs s USING (id)"
            if gb_sql:
                qso += f" GROUP BY {gb_sql}"
        else:

            qo = f"SELECT {final_select} FROM owner.patients_owner o"
            if gb_sql:
                qo += f" GROUP BY {gb_sql}"

    elif strategy == "server-only":

        needs_owner = bool(((sel_plain | gb | agg_args) & Fo))
        if needs_owner:
            proj_qs = ", ".join(["s.id"] + [f"s.{c}" for c in Aqs])
            qs = f"SELECT {proj_qs} FROM server.patients_server s"
            qso = f"SELECT {final_select} FROM owner.patients_owner o JOIN Rs s USING (id)"
            if gb_sql:
                qso += f" GROUP BY {gb_sql}"
        else:
            gb_only_s = ", ".join([f"s.{c}" for c in sorted(gb & Fs)])
            aggs_sql_s = render_aggs_sql(aggs, Fo)
            parts = []
            if gb_only_s:
                parts.append(gb_only_s)
            if aggs_sql_s:
                parts.append(aggs_sql_s)
            final_s = ", ".join(parts) if parts else aggs_sql_s
            qs = f"SELECT {final_s} FROM server.patients_server s"
            if gb_only_s:
                qs += f" GROUP BY {gb_only_s}"

    elif strategy == "parallel":

        proj_qo = ", ".join(["o.id"] + [f"o.{c}" for c in Aqo]) if Aqo else "o.id"
        qo = f"SELECT {proj_qo} FROM owner.patients_owner o"


        proj_qs = ", ".join(["s.id"] + [f"s.{c}" for c in Aqs]) if Aqs else "s.id"
        qs = f"SELECT {proj_qs} FROM server.patients_server s"


        qso = (
            "SELECT " + final_select +
            " FROM owner.patients_owner o"
            " JOIN Ro r USING (id)"
            " JOIN Rs s USING (id)"
        )
        if gb_sql:
            qso += f" GROUP BY {gb_sql}"

    else:
        raise ValueError("Strategy must be one of: server-owner, owner-server, owner-only, server-only, parallel")

    return qs, qo, qso


In [12]:
def process_query_gb(query: str, Fo: Set[str], Fs: Set[str]) -> Dict[str, any]:

    select_plain, group_by, aggs = parse_query_groupby(query)
    classified_gb = classify_groupby_agg(group_by, aggs, Fo, Fs)
    strategy_key = choose_strategy_groupby(classified_gb, aggs,
                                       fo_table="owner.patients_owner",
                                       fs_table="server.patients_server")
    strategy_eff = strategy_key

    qs, qo, qso = generate_subqueries_gb(
        select_plain=select_plain, group_by=group_by, aggs=aggs,
        Fo=Fo, Fs=Fs, strategy=strategy_eff
    )

    return {
        "Query": query,
        "SELECT_PLAIN": select_plain,
        "GROUP_BY": group_by,
        "AGGS": aggs,
        "Classificazione_GB": classified_gb,
        "Strategia": strategy_key,
        "Strategia_eff": strategy_eff,
        "qs": qs, "qo": qo, "qso": qso
    }


In [13]:
def _replan_alternative_gb(plan: dict, Fo: set, Fs: set) -> dict | None:

    cur = plan.get("Strategia_eff") or plan.get("Strategia")
    alt = {"owner-server": "server-owner", "server-owner": "owner-server"}.get(cur)
    if not alt:
        return None


    qs, qo, qso = generate_subqueries_gb(
        select_plain=plan["SELECT_PLAIN"],
        group_by=plan["GROUP_BY"],
        aggs=plan["AGGS"],
        Fo=Fo, Fs=Fs, strategy=alt
    )
    return {"Strategia": alt, "qs": qs, "qo": qo, "qso": qso}


In [70]:
def evaluate_query_gb(query: str,
                      Fo: set, Fs: set,
                      tag: str | None = None,
                      schema: str = "work",
                      save_to: str | None = None,
                      also_compare_alt: bool = True) -> dict:
    plan = process_query_gb(query, Fo, Fs)
    sk = plan.get("Strategia_eff") or plan["Strategia"]

    tag = tag or uuid.uuid4().hex[:8]

    run(f"CREATE SCHEMA IF NOT EXISTS {schema};")
    ro_name, rs_name, out_name = f"{schema}.ro_{tag}", f"{schema}.rs_{tag}", f"{schema}.out_{tag}"

    counts, sizes = {}, {}

    if sk == "owner-server":
        qo = _strip_semicolon(plan["qo"])
        qs = _strip_semicolon(plan["qs"])
        qso = _strip_semicolon(plan["qso"])

        run(f"DROP TABLE IF EXISTS {ro_name}; CREATE TABLE {ro_name} AS {qo};")
        counts["ro"], sizes["ro"] = _count_table(ro_name), _size_table(ro_name)

        qs_mat = qs.replace(" Ro ", f" {ro_name} ")
        run(f"DROP TABLE IF EXISTS {rs_name}; CREATE TABLE {rs_name} AS {qs_mat};")
        counts["rs"], sizes["rs"] = _count_table(rs_name), _size_table(rs_name)

        qso_mat = qso.replace(" Rs ", f" {rs_name} ")
        run(f"DROP TABLE IF EXISTS {out_name}; CREATE TABLE {out_name} AS {qso_mat};")
        counts["out"], sizes["out"] = _count_table(out_name), _size_table(out_name)

    elif sk == "server-owner":
        qs = _strip_semicolon(plan["qs"])
        qso = _strip_semicolon(plan["qso"])

        run(f"DROP TABLE IF EXISTS {rs_name}; CREATE TABLE {rs_name} AS {qs};")
        counts["rs"], sizes["rs"] = _count_table(rs_name), _size_table(rs_name)

        qso_mat = qso.replace(" Rs ", f" {rs_name} ")
        run(f"DROP TABLE IF EXISTS {out_name}; CREATE TABLE {out_name} AS {qso_mat};")
        counts["out"], sizes["out"] = _count_table(out_name), _size_table(out_name)

    elif sk in ("owner-only", "server-only"):

        if plan["qo"]:
            qo = _strip_semicolon(plan["qo"])
            run(f"DROP TABLE IF EXISTS {ro_name}; CREATE TABLE {ro_name} AS {qo};")
            counts["ro"], sizes["ro"] = _count_table(ro_name), _size_table(ro_name)

        if plan["qs"]:
            qs = _strip_semicolon(plan["qs"])
            qs_mat = qs.replace(" Ro ", f" {ro_name} ") if plan["qo"] else qs
            run(f"DROP TABLE IF EXISTS {rs_name}; CREATE TABLE {rs_name} AS {qs_mat};")
            counts["rs"], sizes["rs"] = _count_table(rs_name), _size_table(rs_name)

        #qso = _strip_semicolon(plan["qso"])
        #qso_mat = qso.replace(" Rs ", f" {rs_name} ") if plan["qs"] else qso
        #run(f"DROP TABLE IF EXISTS {out_name}; CREATE TABLE {out_name} AS {qso_mat};")
        #counts["out"], sizes["out"] = _count_table(out_name), _size_table(out_name)

    elif sk == "parallel":
        # materializza entrambi i lati, poi la query finale che li usa entrambi
        qo = _strip_semicolon(plan["qo"])
        qs = _strip_semicolon(plan["qs"])
        qso = _strip_semicolon(plan["qso"])

        run(f"DROP TABLE IF EXISTS {ro_name}; CREATE TABLE {ro_name} AS {qo};")
        counts["ro"], sizes["ro"] = _count_table(ro_name), _size_table(ro_name)

        run(f"DROP TABLE IF EXISTS {rs_name}; CREATE TABLE {rs_name} AS {qs};")
        counts["rs"], sizes["rs"] = _count_table(rs_name), _size_table(rs_name)

        qso_mat = (
            qso.replace(" Ro ", f" {ro_name} ")
               .replace(" Rs ", f" {rs_name} ")
        )
        run(f"DROP TABLE IF EXISTS {out_name}; CREATE TABLE {out_name} AS {qso_mat};")
        counts["out"], sizes["out"] = _count_table(out_name), _size_table(out_name)

    else:
        raise ValueError(f"Strategia sconosciuta: {sk!r}")

    ship_ro, ship_rs = _ships(plan)
    net_bytes = _network_bytes_payload(
            ro_name if ship_ro else None,
            rs_name if ship_rs else None
    )

    alt_info = None
    if also_compare_alt and sk in ("owner-server", "server-owner"):
        alt = _replan_alternative_gb(plan, Fo, Fs)
        if alt:
            tag_alt = tag + "_alt"
            ro_alt, rs_alt, out_alt = f"{schema}.ro_{tag_alt}", f"{schema}.rs_{tag_alt}", f"{schema}.out_{tag_alt}"
            sizes_alt = {}

            if alt["Strategia"] == "owner-server":
                qo_alt = _strip_semicolon(alt["qo"])
                qs_alt = _strip_semicolon(alt["qs"])
                qso_alt = _strip_semicolon(alt["qso"])

                run(f"DROP TABLE IF EXISTS {ro_alt}; CREATE TABLE {ro_alt} AS {qo_alt};")
                sizes_alt["ro"] = _size_table(ro_alt)

                qs_alt_mat = qs_alt.replace(" Ro ", f" {ro_alt} ")
                run(f"DROP TABLE IF EXISTS {rs_alt}; CREATE TABLE {rs_alt} AS {qs_alt_mat};")
                sizes_alt["rs"] = _size_table(rs_alt)

                qso_alt_mat = qso_alt.replace(" Rs ", f" {rs_alt} ")
                run(f"DROP TABLE IF EXISTS {out_alt}; CREATE TABLE {out_alt} AS {qso_alt_mat};")

            else:  # server-owner
                qs_alt = _strip_semicolon(alt["qs"])
                qso_alt = _strip_semicolon(alt["qso"])
                run(f"DROP TABLE IF EXISTS {rs_alt}; CREATE TABLE {rs_alt} AS {qs_alt};")
                sizes_alt["rs"] = _size_table(rs_alt)
                qso_alt_mat = qso_alt.replace(" Rs ", f" {rs_alt} ")
                run(f"DROP TABLE IF EXISTS {out_alt}; CREATE TABLE {out_alt} AS {qso_alt_mat};")

            ship_ro_alt, ship_rs_alt = _ships(alt)
            net_alt = _network_bytes_payload(
                 ro_alt if ship_ro_alt else None,
                rs_alt if ship_rs_alt else None
                )

            saving_ratio = None
            if net_alt and net_bytes:
                saving_ratio = 1 - (net_bytes / net_alt)
            alt_info = {
                "alt_strategy": alt["Strategia"],
                "alt_network_bytes": net_alt,
                "saving_pct": f"{saving_ratio*100:.1f}%" if saving_ratio is not None else None,
                "tables_alt": {"result_owner": ro_alt if "ro" in sizes_alt else None,
                               "result_server": rs_alt if "rs" in sizes_alt else None,
                               "result_out": out_alt}
            }

    row = {
        "tag": tag,
        "query": plan["Query"],
        "strategy": sk,
        "result_owner": counts.get("ro"), "result_server": counts.get("rs"), "result_out": counts.get("out"),
        "network_bytes": net_bytes,
        "alt_strategy": alt_info["alt_strategy"] if alt_info else None,
        "alt_network_bytes": alt_info["alt_network_bytes"] if alt_info else None,
        "saving_pct": alt_info["saving_pct"] if alt_info else None
    }

    if save_to:
        save_to = os.path.abspath(save_to)
        df = pd.DataFrame([row])
        header = not os.path.exists(save_to)
        df.to_csv(save_to, mode="a", index=False, header=header)

    return {
        "plan": plan,
        "row": row,
        "tables": {"result_owner": ro_name if "ro" in counts else None,
                   "result_server": rs_name if "rs" in counts else None,
                   "result_out": out_name if "out" in counts else None},
        "alt": alt_info
    }


In [71]:
def evaluate_queries_gb(queries: list[str],
                        Fo: set, Fs: set,
                        schema: str = "work",
                        save_to: str | None = None,
                        also_compare_alt: bool = True) -> pd.DataFrame:
    rows = []
    for i, q in enumerate(queries, 1):
        tag = f"hv{i:02d}"
        res = evaluate_query_gb(q, Fo, Fs, tag=tag, schema=schema,
                                save_to=save_to, also_compare_alt=also_compare_alt)
        rows.append(res["row"])
    return pd.DataFrame(rows)

TESTING

In [16]:
def print_simple(res: dict):
    for sec in ("row", "plan", "tables", "alt"):
        d = res.get(sec)
        if isinstance(d, dict) and d:
            print(f"\n[{sec}]")
            w = max(len(k) for k in d)
            for k, v in d.items():
                print(f"{k:<{w}} : {v}")


In [75]:
q = """ SELECT gender, MAX(healthcare_expenses)
FROM patients
GROUP BY gender
    """
res = evaluate_query_gb(q, Fo, Fs, tag="q01")
print(print_simple(res))


[row]
tag               : q01
query             :  SELECT gender, MAX(healthcare_expenses)
FROM patients
GROUP BY gender
    
strategy          : server-only
result_owner      : None
result_server     : 2
result_out        : None
network_bytes     : 0
alt_strategy      : None
alt_network_bytes : None
saving_pct        : None

[plan]
Query              :  SELECT gender, MAX(healthcare_expenses)
FROM patients
GROUP BY gender
    
SELECT_PLAIN       : {'gender'}
GROUP_BY           : {'gender'}
AGGS               : [{'func': 'max', 'arg': 'healthcare_expenses', 'distinct': False, 'alias': None}]
Classificazione_GB : {'G_owner': set(), 'G_server': {'gender'}, 'Agg_owner': set(), 'Agg_server': {'healthcare_expenses'}}
Strategia          : server-only
Strategia_eff      : server-only
qs                 : SELECT s.gender, MAX(s.healthcare_expenses) AS max_healthcare_expenses FROM server.patients_server s GROUP BY s.gender
qo                 : None
qso                : None

[tables]
result_ow

In [52]:
if res["tables"]["result_owner"]:
    print("ro")
    run(f"SELECT * FROM {res['tables']['result_owner']} ;", show=True)  # Ro (Qo)
if res["tables"]["result_server"]:
    print("rs")
    run(f"SELECT * FROM {res['tables']['result_server']} ;", show=True)  # Rs (Qs)
print("rout")
run(f"SELECT * FROM {res['tables']['result_out']};")  # Out (Qso)


rs


,id,gender,race
0,c10e497b-ed25-d6c6-e043-144724bf84da,M,asian
1,08c1e3c5-4732-9008-ddd4-edc1f2358521,M,white
2,ac5294eb-05dc-ed5b-2e7c-021eebd1c7b3,F,white
3,aeabefce-854a-81f8-2a92-134a22ae6871,F,white
4,a514d082-312d-f9cf-6e1b-42b6fa1bfb6f,F,white
...,...,...,...
107,f15015b7-a177-da0a-d208-3c63f799da12,F,white
108,92057acb-1a4e-921b-8d21-822d52094f47,M,white
109,bd38b1a4-d16e-a126-426c-0a4a278d1948,M,white
110,80139337-c548-00de-53c3-5cf9aae9af60,F,white


rout


,gender,race,sum_income
0,F,white,4903119.00
1,M,white,4505781.00
2,M,other,21420.00
3,F,asian,210408.00
4,M,black,99502.00
5,F,black,314643.00
6,M,asian,1113013.00
7,F,hawaiian,73330.00


In [ ]:
queries = [
    "SELECT city, gender, AVG(income) AS avg_inc FROM patients  GROUP BY city, gender",
    "SELECT city, max(healthcare_coverage) AS max_healthcare_coverage FROM patients GROUP BY city",
    "",

]
df = evaluate_queries_gb(queries, Fo, Fs, save_to='query2_evaluation.cvs')
df

In [54]:
q = """ SELECT city, MAX(healthcare_coverage)
FROM patients
WHERE gender = 'F'
GROUP BY  city
HAVING AVG(income) > 100000
    """

print(run(q))

          city         max
0    Cambridge     3211.68
1    Arlington    15184.83
2   Plainville  1222754.40
3     Falmouth  1051226.46
4    Lexington  1242825.93
5       Agawam   906315.92
6     Plymouth  1839059.08
7       Leyden     5426.76
8    Billerica  1087558.13
9       Boston   564681.56
10      Newton    13036.13
11     Ashland   754717.41
